In [ ]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [ ]:
sql = f"""
WITH target_devices AS (
  SELECT
    DeviceId,
    MIN(Time) AS first_takeaway_time
  FROM `openrice-production.ORGA.PV_20260915`
  WHERE DeviceId IS NOT NULL
    AND LOWER(EventLabelRaw) LIKE '%takeaway%'
  GROUP BY DeviceId
  ORDER BY first_takeaway_time
  LIMIT 20
)

SELECT pv.*
FROM `openrice-production.ORGA.PV_20260915` AS pv
INNER JOIN target_devices AS target
  ON pv.DeviceId = target.DeviceId
ORDER BY target.first_takeaway_time, pv.Time;
"""

#Execute query
df_big_query = client.query(sql).result().to_dataframe()
df_big_query